# Paired score and ethnicity figures


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
PROJECT=Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
OUT=PROJECT/'reviewer_revision_figures'
OUT.mkdir(exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none','pdf.fonttype':42})
colors=['#405B88','#009E87']
def save(fig,name):
    fig.savefig(OUT/(name+'.pdf'),bbox_inches='tight')
    fig.savefig(OUT/(name+'.svg'),bbox_inches='tight')
    fig.savefig(OUT/(name+'.png'),dpi=300,bbox_inches='tight')
    plt.show()
# Paired score comparison: source values and 95% CIs only; no model refitting.
p=PROJECT/'runs/reviewer_raw106_score/04_results/score106_vs_score249_valid5/paired_score106_minus_score249.csv'
d=pd.read_csv(p).sort_values('Horizon')
fig,axes=plt.subplots(1,2,figsize=(8.2,3.2))
for ax,metric,title in zip(axes,['Delta_Uno_C','NRI_Continuous'],['A  Discrimination','B  Continuous reclassification']):
    v=d[metric].to_numpy();lo=d[metric+'_Lower95'].to_numpy();hi=d[metric+'_Upper95'].to_numpy()
    ax.errorbar(v,np.arange(len(d)),xerr=[v-lo,hi-v],fmt='o',color=colors[0],capsize=3,lw=1.2)
    ax.axvline(0,color='0.5',ls='--',lw=.8)
    ax.set_yticks(np.arange(len(d)),[f'{h:g} years' for h in d.Horizon]);ax.invert_yaxis();ax.set_ylim(len(d)-.5,-.5)
    ax.set_title(title,loc='left',fontsize=11);ax.set_xlabel('Change in Uno C-index' if metric=='Delta_Uno_C' else 'Continuous NRI')
fig.suptitle('Score106 versus Score249: paired test-set comparison',fontsize=12)
fig.tight_layout(rect=[0,0.08,1,.93]);fig.text(.5,.015,'New: Clinical + PRS + Score106; reference: Clinical + PRS + Score249. Bars: 95% CIs.',ha='center',fontsize=8)
save(fig,'revision_paired_scores_C_and_NRI')
# Two separate panels prevent the imprecise smaller subgroup from compressing the White panel.
frames=[]
for run,label in [('reviewer_full249_score_5fold','Score249'),('reviewer_raw106_score','Score106')]:
    t=pd.read_csv(PROJECT/'runs'/run/'04_results/reviewer_ethnicity_all_prediction_models/all_paired_differences.csv')
    t=t[(t.Horizon==10)&(t.Comparison=='Representation_vs_Clinical_PRS')&t.New.eq('Clinical_plus_PRS_plus_NMR_score__Cox PH')].copy()
    t['Score']=label;frames.append(t)
t=pd.concat(frames,ignore_index=True)
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
for ax,group,title in zip(axes,['White','Known_nonWhite_mixed_other'],['A  White','B  Non-White, mixed or other']):
    u=t[t.Group==group].set_index('Score').loc[['Score249','Score106']]
    for i,(label,row) in enumerate(u.iterrows()):
        ax.errorbar(row.Delta_Uno_C,i,xerr=[[row.Delta_Uno_C-row.Delta_Uno_C_Lower95],[row.Delta_Uno_C_Upper95-row.Delta_Uno_C]],fmt='o',color=colors[i],capsize=3)
    ax.axvline(0,color='0.5',ls='--',lw=.8);ax.set_yticks([0,1],['Score249','Score106']);ax.set_ylim(1.6,-.6)
    ax.set_title(title,loc='left',fontsize=11);ax.set_xlabel('10-year change in Uno C-index')
fig.suptitle('Adding an NMR score to Clinical + PRS',fontsize=12)
fig.tight_layout(rect=[0,.1,1,.93]);fig.text(.5,.015,'Pointwise 95% CIs; exploratory internal subgroups. Panels have different x-axis ranges.',ha='center',fontsize=8)
save(fig,'revision_ethnicity_score_increment')
print('Saved:',OUT)
